# Pamoka 12 - Pokalbių Istorijos Sumažinimas su Agentu Užrašų Knygele

Ši užrašinė demonstruoja, kaip valdyti kontekstą ilguose pokalbiuose naudojant Microsoft Agent Framework. Kadangi pokalbiai ilgėja, didėja ir žodžių skaičius — galiausiai viršijantis modelio konteksto langą. Tai sprendžiame naudodami **konteksto santraukos šabloną** ir **agento užrašų knygelę**, skirtą nuolatinei atminčiai.

## Ko Išmoksite:
1. **Kodėl Konteksto Valdymas Svarbus**: Suprasti žodžių limitus ir konteksto langus
2. **Kontekstui Nepažeidžiami Agentai**: Kurti agentus, kurie valdo savo pokalbių kontekstą
3. **Konteksto Santraukos Šablonas**: Naudoti įrankius pokalbio istorijai sutraukti
4. **Agento Užrašų Knygelė**: Nuolatinė atmintis, išliekanti po konteksto sumažinimo

## Reikalingos Sąlygos:
- Azure OpenAI nustatymai su aplinkos kintamaisiais
- Esminės agentų sąvokos iš ankstesnių pamokų supratimas


## Sąranka


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Kodėl svarbus konteksto valdymas

Kiekvienas LLM turi ribotą **konteksto langą** — didžiausią žetonų skaičių, kurį gali apdoroti per vieną užklausą. Vykstant daugiameniam pokalbiui:

- **Žetonų skaičius auga tiesiškai** su kiekvienu vartotojo pranešimu ir asistento atsakymu.
- **Raginimo žetonai lemia kainą**, nes visa istorija siunčiama iš naujo kiekvieną kartą.
- Galiausiai pokalbis **viršija konteksto langą** ir modelis arba sutrumpina, arba išmeta klaidą.

### Strategijos konteksto valdymui

| Strategija | Kaip veikia | Kompromisas |
|---|---|---|
| **Sutrumpinimas** | Išmeta seniausius pranešimus | Prarandamas ankstesnis kontekstas |
| **Santrauka** | Senesnius pranešimus apibendrina į santrauką | Kai kurios detalės pametamos, bet pagrindinės mintys išlieka |
| **Užrašų bloknotas / Išorinė atmintis** | Svarbią informaciją laikyti už pokalbio ribų | Reikalauja įrankių iškvietimų, bet išlieka nepriklausomai nuo sumažinimo |

Šiame užsiėmime deriname **santrauką** su **užrašų bloknoto įrankiu**, kad agentas galėtų išlaikyti tęstinumą net kai pokalbio istorija sutrumpinama.


## Sukuriant kontekstui jautrų agentą


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simuliuojame ilgą pokalbį

Pažvelkime į daugiažingsnį pokalbį, kad pamatytume, kaip kaupiasi kontekstas. Agentas turėtų išlaikyti svarbias detales (preferencijas, biudžetą, kelionės datas) per pokalbio eigą ir parodyti nuoseklumą.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Atkreipkite dėmesį, kaip agentas išlaiko kontekstą iš ankstesnių pokalbių – jis žino apie Japoniją, suši, šventyklas, fotografiją, 3000 dolerių biudžetą, solo keliones ir balandžio laikotarpį. Trumpame pokalbyje tai veikia gerai, tačiau ilgesniame pokalbyje visas istorijos siuntimas tampa brangus.

Tęskime pokalbį su daugiau įrašų, kad pamatytume, kaip kaupiasi kontekstas:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Konteksto santraukos šablonas

Kai pokalbis ilgėja, galime naudoti **santraukos įrankį**, kad sukauptą kontekstą sutrauktume į kompaktišką formatą. Agentas kviečia šį įrankį įrašyti svarbiausias nuostatas, kad net jei senesnės žinutės būtų pašalintos, esminė informacija būtų išsaugota.

Šis šablonas yra pažangesnio istorijos trumpinimo pagrindas:
1. Agentas identifikuoja pagrindinius faktus iš pokalbio
2. Jis kviečia santraukos įrankį juos išsaugoti
3. Senesnės žinutės gali būti saugiai pašalintos, nes santrauka apima svarbiausią informaciją

Žemiau apibrėžiame `summarize_preferences` įrankį, kurį agentas gali kviesti, norėdamas įrašyti kompaktišką jam žinomų nuostatų santrauką.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Santrauka

Šioje pamokoje sužinojote, kaip valdyti kontekstą ilgai trunkančiuose agentų pokalbiuose naudojant Microsoft Agent Framework:

### Pagrindinės sąvokos
- **Konteksto langai yra apriboti** — kiekvienas žetonas pokalbio istorijoje kainuoja pinigus ir skaičiuojamas į ribą.
- **Apibendrinimo įrankiai** leidžia agentui sutrumpinti sukauptą kontekstą į glaustus santraukas, sumažinant žetonų suvartojimą, bet išlaikant svarbią informaciją.
- **Agentų užrašų bloknotai** suteikia nuolatinę išorinę atmintį, kuri išlieka nepaisant pokalbio sumažinimo.

### Ką sukūrėte
- **Konteksto suvokiantis agentą**, kuris palaiko tęstinumą per kelių žingsnių pokalbius
- **Apibendrinimo įrankį** (`summarize_preferences`), kuris įrašo svarbiausias vartotojo detales glaustu formatu
- **Daugiapartį pokalbį**, demonstruojantį konteksto išlaikymą ir pokyčių valdymą

### Tikro pasaulio taikymai
- **Klientų aptarnavimo botai**: prisimena pageidavimus per ilgus palaikymo sesijų pokalbius
- **Asmeniniai asistentai**: seka vykdomus projektus be būtinybės vėl aiškinti kontekstą
- **Mokomieji vadovai**: palaiko mokinių pažangą per daugybę sąveikų

### Kiti žingsniai
- Įgyvendinti pilną užrašų bloknoto įrankį su failų pagrindu veikiančiu išsaugojimu
- Pridėti automatinį istorijos sutrumpinimą po apibendrinimo
- Integruoti su vektorinėmis duomenų bazėmis semantiniam atminčiai ieškoti
- Kurti agentus, galinčius vėliau tą pačią dieną tęsti pokalbius su pilnu kontekstu


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Dėl svarbios informacijos rekomenduojama naudoti profesionalų žmogaus atliktą vertimą. Mes neprisiimame atsakomybės už bet kokius nesusipratimus ar neteisingus aiškinimus, kylančius naudojant šį vertimą.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
